In [18]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score 


In [19]:
df = pd.read_csv("workplace_productivity.csv")

فقط روی داده های عددی و فاصله اقلیدسی کار میکند  k-means این دیتاست عددی هستش چون الگوریتم 

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   noise_level_db         1000 non-null   float64
 1   light_exposure_lux     1000 non-null   float64
 2   temperature_c          1000 non-null   float64
 3   crowding_index         1000 non-null   float64
 4   meeting_hours          1000 non-null   float64
 5   remote_work_pct        1000 non-null   float64
 6   green_space_score      1000 non-null   float64
 7   break_area_score       1000 non-null   float64
 8   team_seniority         1000 non-null   int64  
 9   project_complexity     1000 non-null   float64
 10  deadline_pressure      1000 non-null   float64
 11  team_size              1000 non-null   int64  
 12  manager_support_score  1000 non-null   float64
 13  training_hours         1000 non-null   float64
 14  productivity_score     1000 non-null   float64
dtypes: float64(13), 

استفاده میکنیم info برای نمایش اطلاعات کلی دیتاست از 

In [21]:
df.head()

,noise_level_db,light_exposure_lux,temperature_c,crowding_index,meeting_hours,remote_work_pct,green_space_score,break_area_score,team_seniority,project_complexity,deadline_pressure,team_size,manager_support_score,training_hours,productivity_score
0,66.971340,217.507529,21.300352,3.008897,18.411780,67.669949,9.029616,1.782449,14,1.286044,1.843257,10,5.548198,1.061439,50.824219
1,62.247074,354.308435,25.071188,8.284874,0.162469,80.581925,7.283255,4.062255,5,2.937824,7.871447,6,1.834713,3.868655,48.375369
2,77.374718,622.608222,27.685539,7.567586,13.405702,97.311576,4.406809,5.968366,20,8.969066,4.254715,9,7.341147,1.832975,46.649201
3,38.989599,362.953620,19.212017,3.501762,15.892111,36.483218,4.331629,2.885563,9,7.316383,7.151929,5,6.482179,6.845546,51.500450
4,71.456340,314.381746,22.553465,9.905710,15.999994,55.694974,7.161528,8.585667,2,3.061433,1.288902,13,4.610483,2.647544,51.593058


| ستون                      | توضیح                               |
| ------------------------- | ----------------------------------- |
| **noise_level_db**        | میزان سر و صدای محیط کار            |
| **light_exposure_lux**    | شدت نور محیط کار                    |
| **temperature_c**         | دمای محیط کار                       |
| **crowding_index**        | میزان شلوغی محیط کار                |
| **meeting_hours**         | تعداد ساعات جلسات                   |
| **remote_work_pct**       | درصد دورکاری                        |
| **green_space_score**     | امتیاز فضای سبز محیط کار            |
| **break_area_score**      | امتیاز کیفیت فضای استراحت           |
| **team_seniority**        | میزان سابقه و تجربه تیم             |
| **project_complexity**    | میزان پیچیدگی پروژه                 |
| **deadline_pressure**     | میزان فشار زمانی برای تحویل پروژه   |
| **team_size**             | تعداد اعضای تیم                     |
| **manager_support_score** | میزان حمایت مدیر                    |
| **training_hours**        | ساعات آموزش کارکنان                 |
| **productivity_score**    | امتیاز بهره‌وری کارکنان (متغیر هدف) |


In [22]:
df.describe()

,noise_level_db,light_exposure_lux,temperature_c,crowding_index,meeting_hours,remote_work_pct,green_space_score,break_area_score,team_seniority,project_complexity,deadline_pressure,team_size,manager_support_score,training_hours,productivity_score
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,59.800128,555.668567,24.080151,5.389855,12.872217,49.627560,5.640292,5.476277,10.550000,5.454825,5.526893,14.044000,5.530549,19.424876,51.326379
std,14.301065,199.830503,3.462817,2.634983,7.189341,28.667790,2.567778,2.632047,5.767388,2.595322,2.624534,6.467758,2.559731,11.632884,7.228372
min,35.123654,201.712672,18.004243,1.002222,0.002309,0.084321,1.007590,1.029404,1.000000,1.020254,1.018785,3.000000,1.009058,0.021717,29.029494
25%,47.791318,388.312902,21.162235,3.030789,6.634173,26.405181,3.460830,3.138713,5.000000,3.265313,3.201515,9.000000,3.323193,8.703241,46.236213
50%,59.517792,556.723761,24.145126,5.208255,13.153774,48.399129,5.726792,5.392147,11.000000,5.364371,5.578063,14.000000,5.512659,18.784941,51.418769
75%,72.126263,733.251294,27.064573,7.710881,19.089892,73.912737,7.803636,7.809795,16.000000,7.710055,7.689592,20.000000,7.726697,29.626285,56.306181
max,84.889926,899.174877,29.994869,9.981671,24.945324,99.778712,9.996148,9.993542,20.000000,9.999588,9.998390,25.000000,9.999170,39.977175,73.693952



استفاده میکنیم describe برای بدست اوردن آمار توصیفی ستون های عددی دیتاست از 

In [23]:
df.isnull().sum()

noise_level_db           0
light_exposure_lux       0
temperature_c            0
crowding_index           0
meeting_hours            0
remote_work_pct          0
green_space_score        0
break_area_score         0
team_seniority           0
project_complexity       0
deadline_pressure        0
team_size                0
manager_support_score    0
training_hours           0
productivity_score       0
dtype: int64

استفاده میکنیم  isnull برای بررسی مقادیر گمشده در هر ستون دیتاست از 

In [ ]:
df[df.duplicated(keep=False)]
df.drop_duplicates()



,noise_level_db,light_exposure_lux,temperature_c,crowding_index,meeting_hours,remote_work_pct,green_space_score,break_area_score,team_seniority,project_complexity,deadline_pressure,team_size,manager_support_score,training_hours,productivity_score


 رکورد های تکراری را بررسی و حذف کردیم تا این رکورد تکراری باعث خراب شدن دیتاست نشود